![Week 4 Day 1](../../assets/banners/w4d1.png)

# Week 4 Day 1 - Solution: An Adaptive Router for the Toy Graph

The exercise: add a `route` step that sends greetings straight to a friendly reply, and everything else to the search. Then see what `MAX_REWRITES = 0` changes. This is the **adaptive** part of the agentic RAG spectrum, in plain Python.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Try the exercise yourself before reading on: struggling a little is where the learning happens! These are <i>reference</i> solutions, not the only right answers.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

First, the toy pieces from the lab, unchanged:

In [ ]:
import re
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

NOTES = {
    "1.4 CPU scheduling": "The scheduler picks which ready process runs next. Round robin gives each one a time slice.",
    "1.6 Deadlocks": "A deadlock is a set of processes each waiting for a resource held by another. "
    "Four conditions: mutual exclusion, hold and wait, no preemption, circular wait.",
    "2.5 Transport layer": "TCP gives reliable, ordered delivery. UDP is connectionless and faster.",
    "3.4 Transactions": "ACID: atomicity, consistency, isolation and durability.",
}
STOPWORDS = {
    "a",
    "an",
    "the",
    "is",
    "are",
    "my",
    "why",
    "does",
    "do",
    "what",
    "when",
    "who",
    "won",
    "of",
    "and",
    "each",
}
GLOSSARY = {"freeze": "deadlock", "stuck": "deadlock", "hang": "deadlock", "handshake": "TCP", "fast": "UDP"}


def words(text: str) -> set[str]:
    return {w.rstrip("s") for w in re.findall(r"[a-z0-9]+", text.lower())} - STOPWORDS


def search(query: str, k: int = 1) -> list[str]:
    scored = sorted(((len(words(query) & words(text)), title) for title, text in NOTES.items()), reverse=True)
    return [title for score, title in scored[:k] if score > 0]


class MiniState(TypedDict):
    question: str
    query: str
    docs: list[str]
    rewrites: int
    answer: str

### The router

💡 The trick is a **conditional entry point**: `add_conditional_edges(START, route)`. The router is a plain function, like `grade`, and its `Literal` return type tells LangGraph (and the diagram) where it can go. A real system would ask a small LLM "does this need the course notes?"; a keyword check is enough to show the shape.

In [ ]:
from gradewise.display import show_graph

SMALL_TALK = re.compile(r"^\s*(hi|hello|hey|thanks|thank you|good (morning|afternoon|evening))\b", re.I)


def route(state: MiniState) -> Literal["chitchat", "retrieve"]:
    return "chitchat" if SMALL_TALK.search(state["question"]) else "retrieve"


def chitchat(state: MiniState) -> dict:
    return {"answer": "Hi! 👋 Ask me anything about CS201 and I'll find it in the notes."}


def build(max_rewrites: int = 2):
    def retrieve(state: MiniState) -> dict:
        return {"docs": search(state["query"])}

    def grade(state: MiniState) -> Literal["generate", "rewrite"]:
        return "generate" if state["docs"] or state["rewrites"] >= max_rewrites else "rewrite"

    def rewrite(state: MiniState) -> dict:
        query = " ".join(GLOSSARY.get(w.rstrip("?.,!").lower(), w) for w in state["query"].split())
        return {"query": query, "rewrites": state["rewrites"] + 1}

    def generate(state: MiniState) -> dict:
        if not state["docs"]:
            return {"answer": "The course notes don't cover that."}
        return {"answer": f"{NOTES[state['docs'][0]]} [{state['docs'][0]}]"}

    g = StateGraph(MiniState)
    for node in (chitchat, retrieve, rewrite, generate):
        g.add_node(node.__name__, node)
    g.add_conditional_edges(START, route)  # 💡 the adaptive part
    g.add_conditional_edges("retrieve", grade)
    g.add_edge("rewrite", "retrieve")
    g.add_edge("chitchat", END)
    g.add_edge("generate", END)
    return g.compile()


adaptive = build()
show_graph(adaptive)

In [ ]:
def ask(graph, question: str) -> None:
    s = graph.invoke({"question": question, "query": question, "docs": [], "rewrites": 0, "answer": ""})
    print(f"❓ {question}\n   rewrites: {s['rewrites']} · 💬 {s['answer'][:100]}\n")


for q in [
    "Hi there!",
    "Thanks for the help",
    "What are the four conditions for deadlock?",
    "Why does my program freeze forever?",
]:
    ask(adaptive, q)

The diagram now has two dashed edges out of `__start__`: one to `chitchat`, one to `retrieve`. Greetings never touch the search, which is exactly what GradeWise's `generate_query_or_respond` node does when the model decides not to call the tool.

### Bonus: `MAX_REWRITES = 0`

In [ ]:
ask(build(max_rewrites=0), "Why does my program freeze forever?")

With a cap of 0, `grade` goes straight to `generate` after the first (failed) search, so the loop never gets the chance to swap *freeze* for *deadlock*: the student is told the notes don't cover it, even though they do. That's plain 2-step RAG again. The cap is a trade-off: **0** is cheapest but brittle, **1–2** fixes most vocabulary mismatches, and anything higher mostly burns tokens on questions the notes genuinely can't answer.